In [ ]:
# Check GPU
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Install
!pip install -q diffusers transformers accelerate
print("✓ Installed")

In [ ]:
# Load DiT-XL
from diffusers import DiTPipeline, DPMSolverMultistepScheduler
import torch

print("Loading DiT-XL/2-256 (~2GB download)...")

pipe = DiTPipeline.from_pretrained(
    "facebook/DiT-XL-2-256",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = pipe.to(device)

print(f"✓ Loaded on {device}")

## Generate Images

DiT uses ImageNet class IDs (0-999):
- 207 = Golden Retriever
- 281 = Tabby Cat
- 388 = Giant Panda

In [ ]:
# Generate one image
from IPython.display import display

class_id = 207  # Golden Retriever

image = pipe(
    class_labels=[class_id],
    num_inference_steps=25,
    generator=torch.Generator(device=device).manual_seed(42)
).images[0]

display(image)
print(f"Generated class {class_id}")

In [ ]:
# Generate multiple
classes = {207: "Golden Retriever", 281: "Tabby Cat", 388: "Giant Panda"}

for cid, name in classes.items():
    print(f"\n[{cid}] {name}")
    img = pipe(
        class_labels=[cid],
        num_inference_steps=25,
        generator=torch.Generator(device=device).manual_seed(cid)
    ).images[0]
    display(img)